# 07 — LLM Context Classification Results

Visualises the output of `06_llm_context_analysis.py`, which used `claude-haiku-4-5-20251001` to classify the geoeconomic-risk context in each of the 250 flagged transcripts.

**Eight questions were asked for each transcript:**
| Field | Question |
|---|---|
| `firm_operations_relevance` | Is the risk tied to this firm's own revenues, costs, supply chain, or operations? |
| `macro_context` | Is the risk placed in a broader macroeconomic or industry-wide frame? |
| `speaker_attribution` | Who raises the topic — executives only, analysts only, or both? |
| `response_discussed` | Does management describe concrete actions in response? |
| `increase_investments` | Increasing capex or expanding capacity as a response? |
| `decrease_investments` | Cutting or deferring investment as a response? |
| `find_new_suppliers` | Diversifying supply base or sourcing from alternative regions? |
| `stop_exports` | Halting or reducing exports or withdrawing from affected markets? |

In [1]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from pathlib import Path

# ── Load data ──────────────────────────────────────────────────────────────────
with open('data/geoeconomic_context.json') as f:
    raw = json.load(f)

df = pd.DataFrame(raw)
df = df[df.get('error', pd.Series([None]*len(df))).isna()] if 'error' in df.columns else df

# Normalise boolean columns (LLM sometimes returns strings)
bool_cols = ['firm_operations_relevance', 'macro_context', 'response_discussed',
             'increase_investments', 'decrease_investments', 'find_new_suppliers', 'stop_exports']
for c in bool_cols:
    df[c] = df[c].map(lambda x: bool(x) if not isinstance(x, str) else x.lower() == 'true')

# Explode risk_types list → one row per risk category per document
df_risk = df.explode('risk_types').copy()

print(f'Loaded {len(df):,} classified transcripts')
print(f'Has reporting period: {df["reporting_period"].notna().sum()} docs')
df[bool_cols].mean().round(3)

Loaded 250 classified transcripts
Has reporting period: 250 docs


firm_operations_relevance    0.696
macro_context                0.720
response_discussed           0.512
increase_investments         0.116
decrease_investments         0.056
find_new_suppliers           0.100
stop_exports                 0.016
dtype: float64

In [2]:
# ── Shared style ───────────────────────────────────────────────────────────────

BLUE   = '#2171B5'
GREY   = '#AAAAAA'
DARK   = '#333333'
ACCENT = '#E6550D'

PALETTE_ATTR = {'executive_only': '#2171B5', 'analyst_only': '#E6550D', 'both': '#31A354'}

def save(fig, name, tight=True):
    Path('figures').mkdir(exist_ok=True)
    if tight:
        fig.tight_layout()
    fig.savefig(f'figures/{name}', dpi=160, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f'Saved figures/{name}')

plt.rcParams.update({
    'font.family': 'sans-serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.labelcolor': DARK,
    'xtick.color': DARK,
    'ytick.color': DARK,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
})

## Figure 19 — Binary flag overview

In [3]:
labels = {
    'firm_operations_relevance': 'Firm-level\noperations relevance',
    'macro_context':             'Macro / industry\nframing',
    'response_discussed':        'Concrete management\nresponse mentioned',
    'increase_investments':      'Increase capex\n/ inventory',
    'decrease_investments':      'Cut / defer\ninvestment',
    'find_new_suppliers':        'Diversify\nsupply base',
    'stop_exports':              'Halt / reduce\nexports',
}

pcts = {k: df[k].mean() * 100 for k in labels}
sorted_items = sorted(pcts.items(), key=lambda x: x[1], reverse=True)
keys   = [k for k, _ in sorted_items]
vals   = [v for _, v in sorted_items]
xlabels = [labels[k] for k in keys]

colors = [BLUE if v >= 40 else GREY for v in vals]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(range(len(keys)), vals, color=colors, height=0.65, zorder=3)
ax.set_yticks(range(len(keys)))
ax.set_yticklabels(xlabels, fontsize=10)
ax.set_xlabel('Share of flagged transcripts (%)', fontsize=10)
ax.set_xlim(0, 105)
ax.axvline(50, color='#CCCCCC', lw=0.8, zorder=2)
ax.grid(axis='x', color='#EBEBEB', lw=0.5, zorder=0)

for i, v in enumerate(vals):
    ax.text(v + 1.5, i, f'{v:.0f}%', va='center', fontsize=9, color=DARK)

ax.set_title('LLM Context Classification: Binary Flag Overview',
             fontsize=12, fontweight='bold', loc='left', pad=10)
ax.set_ylabel('')
ax.annotate(f'n = {len(df)} flagged transcripts | Model: claude-haiku-4-5-20251001',
            xy=(0, -0.08), xycoords='axes fraction',
            fontsize=8, color='#888888', style='italic')

save(fig, '19_llm_binary_flags.png')

Saved figures/19_llm_binary_flags.png


## Figure 20 — Speaker attribution

In [4]:
attr_counts = df['speaker_attribution'].value_counts()
# Ensure canonical order
order = ['executive_only', 'analyst_only', 'both']
attr_counts = attr_counts.reindex(order).fillna(0).astype(int)

attr_labels = {'executive_only': 'Executives only', 'analyst_only': 'Analysts only', 'both': 'Both'}
colors_attr = [PALETTE_ATTR[k] for k in order]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5))

# Left: bar chart with counts
bars = ax1.bar([attr_labels[k] for k in order], attr_counts.values,
               color=colors_attr, width=0.55, zorder=3)
ax1.set_ylabel('Number of transcripts', fontsize=10)
ax1.grid(axis='y', color='#EBEBEB', lw=0.5, zorder=0)
for bar, v in zip(bars, attr_counts.values):
    ax1.text(bar.get_x() + bar.get_width() / 2, v + 1,
             f'{v}\n({v/len(df)*100:.0f}%)', ha='center', va='bottom', fontsize=9)
ax1.set_title('Speaker Attribution', fontsize=11, fontweight='bold', loc='left')

# Right: stacked bar per risk type
risk_attr = df_risk.groupby(['risk_types', 'speaker_attribution']).size().unstack(fill_value=0)
risk_attr = risk_attr.reindex(columns=order, fill_value=0)
risk_attr_pct = risk_attr.div(risk_attr.sum(axis=1), axis=0) * 100

bottom = np.zeros(len(risk_attr_pct))
for col, color in zip(order, colors_attr):
    vals_col = risk_attr_pct[col].values if col in risk_attr_pct.columns else np.zeros(len(risk_attr_pct))
    ax2.barh(range(len(risk_attr_pct)), vals_col, left=bottom, color=color, label=attr_labels[col])
    bottom += vals_col

ax2.set_yticks(range(len(risk_attr_pct)))
ax2.set_yticklabels(risk_attr_pct.index.tolist(), fontsize=10)
ax2.set_xlabel('Share (%)', fontsize=10)
ax2.set_xlim(0, 100)
ax2.legend(loc='lower right', fontsize=9)
ax2.set_title('Speaker Attribution by Risk Category', fontsize=11, fontweight='bold', loc='left')

fig.suptitle(f'n = {len(df)} flagged transcripts', fontsize=9,
             color='#888888', y=-0.01, style='italic')
save(fig, '20_llm_speaker_attribution.png')

Saved figures/20_llm_speaker_attribution.png


## Figure 21 — Response types among transcripts where management responded

In [5]:
df_resp = df[df['response_discussed']].copy()

resp_types = ['increase_investments', 'decrease_investments', 'find_new_suppliers', 'stop_exports']
resp_labels = ['Increase capex\n/ inventory', 'Cut / defer\ninvestment',
               'Diversify\nsupply base', 'Halt / reduce\nexports']
resp_colors = ['#2171B5', '#E6550D', '#31A354', '#756BB1']

counts_resp = [df_resp[c].sum() for c in resp_types]
pcts_resp   = [v / len(df_resp) * 100 for v in counts_resp]

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(resp_labels, pcts_resp, color=resp_colors, width=0.55, zorder=3)
ax.set_ylabel('Share of "response discussed" transcripts (%)', fontsize=10)
ax.set_ylim(0, max(pcts_resp) * 1.25 + 5)
ax.grid(axis='y', color='#EBEBEB', lw=0.5, zorder=0)

for bar, v, pct in zip(bars, counts_resp, pcts_resp):
    ax.text(bar.get_x() + bar.get_width() / 2, pct + 0.8,
            f'{v}\n({pct:.0f}%)', ha='center', va='bottom', fontsize=9)

ax.set_title('Management Response Types', fontsize=12, fontweight='bold', loc='left', pad=10)
ax.annotate(f'Subset: {len(df_resp)} transcripts where management described a concrete response\n'
            f'(out of {len(df)} total flagged transcripts; response_discussed = True)',
            xy=(0, -0.18), xycoords='axes fraction',
            fontsize=8, color='#888888', style='italic')

save(fig, '21_llm_response_types.png')

Saved figures/21_llm_response_types.png


## Figure 22 — Risk category × context heatmap

In [6]:
context_flags = ['firm_operations_relevance', 'macro_context', 'response_discussed',
                 'increase_investments', 'decrease_investments', 'find_new_suppliers', 'stop_exports']
flag_labels = {
    'firm_operations_relevance': 'Firm-level relevance',
    'macro_context':             'Macro framing',
    'response_discussed':        'Response discussed',
    'increase_investments':      'Increase investments',
    'decrease_investments':      'Decrease investments',
    'find_new_suppliers':        'Find new suppliers',
    'stop_exports':              'Stop / reduce exports',
}

# Mean of each flag per risk category
hm = df_risk.groupby('risk_types')[context_flags].mean() * 100
hm.columns = [flag_labels[c] for c in context_flags]

# Sort risk categories by total document count
risk_order = df_risk['risk_types'].value_counts().index.tolist()
hm = hm.reindex(risk_order)

fig, ax = plt.subplots(figsize=(10, max(3, len(hm) * 0.8 + 1.5)))
im = ax.imshow(hm.values, cmap='Blues', vmin=0, vmax=100, aspect='auto')

ax.set_xticks(range(len(hm.columns)))
ax.set_xticklabels(hm.columns, rotation=30, ha='right', fontsize=9)
ax.set_yticks(range(len(hm)))
ax.set_yticklabels(hm.index.tolist(), fontsize=10)

# Annotate cells
for i in range(len(hm)):
    for j in range(len(hm.columns)):
        v = hm.values[i, j]
        color = 'white' if v > 60 else '#333333'
        ax.text(j, i, f'{v:.0f}%', ha='center', va='center', fontsize=8, color=color)

plt.colorbar(im, ax=ax, label='Share of documents (%)', fraction=0.025, pad=0.02)
ax.set_title('Geoeconomic Risk Category × Context Flag Heatmap',
             fontsize=12, fontweight='bold', loc='left', pad=10)
ax.annotate('Values: % of documents in each risk category where the flag is True.',
            xy=(0, -0.08 - 0.015 * len(hm.columns)),
            xycoords='axes fraction', fontsize=8, color='#888888', style='italic')

save(fig, '22_llm_risk_context_heatmap.png')

Saved figures/22_llm_risk_context_heatmap.png


## Figure 23 — Firm relevance vs. macro framing by reporting period

In [7]:
PERIOD_ORDER = ['Q2 2025', 'Q3 2025', 'Q4 2025',
                'Q1 2026', 'Q2 2026', 'Q3 2026', 'Q4 2026']

df_period = df[df['reporting_period'].notna() & (df['reporting_period'] != 'nan')].copy()
df_period['reporting_period'] = pd.Categorical(
    df_period['reporting_period'], categories=PERIOD_ORDER, ordered=True
)

period_stats = df_period.groupby('reporting_period', observed=True)[['firm_operations_relevance', 'macro_context']].mean() * 100
period_n     = df_period.groupby('reporting_period', observed=True).size().rename('n')
period_stats = period_stats.join(period_n)

if len(period_stats) < 2:
    print('Not enough reporting-period data for this chart.')
else:
    x = np.arange(len(period_stats))
    w = 0.35
    xlabels = [f'{p}\n(n={n})' for p, n in zip(period_stats.index, period_stats['n'])]

    fig, ax = plt.subplots(figsize=(max(8, len(period_stats) * 1.5), 5))
    ax.bar(x - w/2, period_stats['firm_operations_relevance'], w,
           label='Firm-level relevance', color=BLUE, zorder=3)
    ax.bar(x + w/2, period_stats['macro_context'], w,
           label='Macro framing', color=ACCENT, zorder=3)
    ax.set_xticks(x)
    ax.set_xticklabels(xlabels, fontsize=9, ha='center')
    ax.set_ylabel('Share of transcripts (%)', fontsize=10)
    ax.set_ylim(0, 115)
    ax.grid(axis='y', color='#EBEBEB', lw=0.5, zorder=0)
    ax.legend(fontsize=10)
    ax.set_title('Firm Relevance vs. Macro Framing by Reporting Period',
                 fontsize=12, fontweight='bold', loc='left', pad=10)
    ax.annotate('All quarters with at least one flagged transcript shown. n= labels indicate sample size per period.\n'
                'Cross-quarter comparisons are illustrative only (corpus dominated by Q4 2025).',
                xy=(0, -0.18), xycoords='axes fraction',
                fontsize=8, color='#888888', style='italic')

    save(fig, '23_llm_relevance_by_period.png')

Saved figures/23_llm_relevance_by_period.png


## Figure 24 — Top companies by risk-type depth

In [8]:
# Depth score: sum of all binary flags per company
df_named = df[df['ticker'].notna() & (df['ticker'] != 'nan') & df['company_name'].notna()].copy()
df_named['depth_score'] = df_named[bool_cols].sum(axis=1)

top = (df_named.sort_values('depth_score', ascending=False)
       .head(20)[['company_name', 'ticker', 'depth_score'] + bool_cols]
       .reset_index(drop=True))

fig, ax = plt.subplots(figsize=(10, 7))

y = np.arange(len(top))
colors_depth = [BLUE if s >= 4 else ('#6BAED6' if s >= 3 else GREY) for s in top['depth_score']]

ax.barh(y, top['depth_score'], color=colors_depth, height=0.7, zorder=3)
ax.set_yticks(y)
ax.set_yticklabels([f"{row['company_name']} ({row['ticker']})" for _, row in top.iterrows()], fontsize=9)
ax.set_xlabel('Context depth score (sum of 7 binary flags)', fontsize=10)
ax.set_xlim(0, 8)
ax.grid(axis='x', color='#EBEBEB', lw=0.5, zorder=0)

for i, (_, row) in enumerate(top.iterrows()):
    ax.text(row['depth_score'] + 0.1, i, str(int(row['depth_score'])),
            va='center', fontsize=9, color=DARK)

ax.set_title('Top 20 Companies by Geoeconomic Risk Discussion Depth',
             fontsize=12, fontweight='bold', loc='left', pad=10)
ax.annotate('Depth score = number of True flags across the 7 binary context questions\n'
            '(excludes speaker_attribution). Higher = more detailed and action-oriented disclosure.',
            xy=(0, -0.11), xycoords='axes fraction',
            fontsize=8, color='#888888', style='italic')

save(fig, '24_llm_top_companies_depth.png')

Saved figures/24_llm_top_companies_depth.png


In [9]:
print('All figures saved to figures/.')
print('\nSummary statistics:')
print(df[bool_cols].mean().mul(100).round(1).to_string())
print('\nSpeaker attribution:')
print(df['speaker_attribution'].value_counts(normalize=True).mul(100).round(1))

All figures saved to figures/.

Summary statistics:
firm_operations_relevance    69.6
macro_context                72.0
response_discussed           51.2
increase_investments         11.6
decrease_investments          5.6
find_new_suppliers           10.0
stop_exports                  1.6

Speaker attribution:
speaker_attribution
executive_only    48.8
both              48.0
analyst_only       3.2
Name: proportion, dtype: float64
